# Cell-ECM mechanical reciprocity (Merks 3D CPM-ECM, reproduced in 2D with the real engine)

_Investigation `merks-ecm-reciprocity-2d` — coder reproduction notebook._

**Question.** Can we reproduce the cell-ECM mechanical-reciprocity phenomena of Keijzer &
Merks 2026 (arXiv:2609.02375, a 3D hybrid cellular Potts model coupled to a
discrete deformable fiber network — code unreleased) by driving the REAL
published 2D ancestor engine (Tsingos et al. 2023 static-adhesion TST-MD)
on ordinary hardware?

A real-engine, 2D reproduction of the mechanical-reciprocity phenomena in a
recent (unreleased) 3D cell-ECM model from the Merks lab. The deliverable
has two parts: (1) viva-tstmd — a reusable process-bigraph bridge that runs
the genuine TST-MD simulator (TST C++ + HOOMD + MUSCLE3) CPU-only inside
Docker, with 19 tests; and (2) this investigation, which uses it to measure
the paper's observables on real runs. It is framed deliberately as a
mechanism-level reproduction, not a quantitative figure match.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-cpm/viva-cpm').is_dir():
    REPO = Path('/home/runner/work/viva-cpm/viva-cpm')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_cpm_studies.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Cell contraction & fiber reorientation (real TST-MD, 2D) (`tstmd-contraction-reorientation`)

**Question.** In the real 2D static-adhesion hybrid CPM + bead-spring ECM engine (Tsingos et al. 2023, the published ancestor of Merks' 3D model), does a contractile cell contract against a crosslinked ECM and reorient the surrounding fibers toward itself, reproducing the direction of the 3D paper's remodeling result?

**Objective.** Run one seeded coupled simulation (single contractile cell, 1200 fiber strands, ~3750 crosslinks, 300 MCS) through the real engine and measure the cell-area and reorientation (q/q0) trajectories plus start/end snapshots.

**Hypothesis.** A contractile cell coupled to the fiber network via static adhesions will contract over Monte Carlo time and drag/align nearby fibers toward it, so the fraction of radially-aligned fibers (q) rises above its initial value (q/q0 > 1), as in Keijzer & Merks 2026 Fig 4.

**Claim.** The real engine reproduces, in 2D, the direction of the 3D paper's mechanical-reciprocity result: a contractile cell contracts (13135 -> ~10 lattice sites over ~80 MCS) and reorients fibers toward itself (q/q0 rises 1.00 -> ~1.09 and holds).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_cpm_studies.composites.tstmd.tstmd_coupled` | 0 | n_cross=5000, fiber_stiffness=200.0, target_area=10, mcs=300 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_cpm_studies.composites.tstmd.tstmd_coupled`** — `spec_viva_cpm_studies_composites_tstmd_tstmd_coupled` (a plain, editable dict)


_composite spec file for `viva_cpm_studies.composites.tstmd.tstmd_coupled` not found under `viva_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: tstmd-contraction-reorientation ===
STUDY = 'tstmd-contraction-reorientation'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Contraction & reorientation — interactive**


In [ ]:
# Contraction & reorientation — interactive
show_viz(_render_one('local:TstmdContractionReorientation', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Contractile cell contracts against the ECM | cell_area | final cell_area < 0.10 * initial cell_area |
| Fibers reorient toward the contracting cell (q/q0 > 1) | q_ratio | final q_ratio >= 1.05 AND max q_ratio >= 1.05 |


## Study: Crosslinking builds ECM network connectivity (real TST-MD, 2D) (`tstmd-network-connectivity`)

**Question.** In the real 2D bead-spring ECM engine, does increasing the number of crosslinks raise the connectivity (giant component) of the fiber network, reproducing the percolation mechanism that underlies Fig 2B of Keijzer & Merks 2026?

**Objective.** Sweep the requested crosslink count over {0, 500, 2000, 5000, 10000, 20000} through the real engine and measure, from the built network, the giant-component fraction (beads nodes; fiber bonds + crosslinks edges).

**Hypothesis.** With no crosslinks the fiber network is essentially disconnected; adding crosslinks sharply raises the fraction of beads in the largest connected component, a percolation-onset signature.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_cpm_studies.composites.tstmd.tstmd_coupled` | 0 | n_cross=5000, fiber_stiffness=200.0, target_area=10, mcs=20 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_cpm_studies.composites.tstmd.tstmd_coupled`** — `spec_viva_cpm_studies_composites_tstmd_tstmd_coupled` (a plain, editable dict)


_composite spec file for `viva_cpm_studies.composites.tstmd.tstmd_coupled` not found under `viva_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: tstmd-network-connectivity ===
STUDY = 'tstmd-network-connectivity'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Network connectivity — interactive**


In [ ]:
# Network connectivity — interactive
show_viz(_render_one('local:TstmdNetworkConnectivity', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Crosslinking connects the fiber network (percolation onset) | giant_component | giant_component at the lowest nonzero crosslink setting >= 10 * giant_component at zero crosslinks |


## Open decisions
- Invest in physical->ymmsl unit-scaling calibration to reproduce the quantitative Fig 2/3 equilibrium figures?
